<a href="https://colab.research.google.com/github/Ironwin-15/FPGA/blob/main/FPGA_PYTHON_SCRIPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from tensorflow import keras

In [ ]:
# Load dataset
iris = load_iris()
X = iris.data      # 4 features
y = iris.target    # labels (0,1,2)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(8, activation='relu', input_shape=(4,)),
    keras.layers.Dense(3, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(X_train, y_train, epochs=400, verbose=1)
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Epoch 1/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 203ms/step - accuracy: 0.3333 - loss: 1.6069
Epoch 2/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.5658 
Epoch 3/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3333 - loss: 1.5278 
Epoch 4/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.4870 
Epoch 5/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.4533 
Epoch 6/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.4212 
Epoch 7/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.3903 
Epoch 8/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.3632 
Epoch 9/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.3378 
Epoch 10/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.3146 
Epoch 11/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3333 - loss: 1.2904 
Epoch 12/400
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3250 - lo

In [ ]:
import os

# ==============================
# 1. Create weights folder
# ==============================
os.makedirs("weights", exist_ok=True)

# ==============================
# 2. Quantization function
# ==============================
def quantize(x):
    q = int(round(x * 256))
    return max(-32768, min(32767, q))

# ==============================
# 3. Extract weights
# ==============================
weights = model.get_weights()

# weights[0] → (4,8)
# weights[1] → (8,)
# weights[2] → (8,3)
# weights[3] → (3,)

# ==============================
# 4. Save WEIGHTS (flattened)
# ==============================
all_weights = []

for w in weights:
    if len(w.shape) > 1:  # only matrices
        flat = w.flatten()
        for val in flat:
            all_weights.append(quantize(val))

with open("weights/weights.mem", "w") as f:
    for val in all_weights:
        if val < 0:
            val = (1 << 16) + val
        f.write(f"{val:04x}\n")

# ==============================
# 5. Save BIASES
# ==============================
all_biases = []

# hidden biases
for val in weights[1]:
    all_biases.append(quantize(val))

# output biases
for val in weights[3]:
    all_biases.append(quantize(val))

with open("weights/biases.mem", "w") as f:
    for val in all_biases:
        if val < 0:
            val = (1 << 16) + val
        f.write(f"{val:04x}\n")

# ==============================
# 6. Save TEST DATA (10 samples)
# ==============================
with open("weights/test_data.mem", "w") as f:
    for i in range(10):
        sample = X_test[i]
        label = y_test[i]

        # save input features
        for val in sample:
            q = quantize(val)
            if q < 0:
                q = (1 << 16) + q
            f.write(f"{q:04x}\n")

        # save expected output
        f.write(f"{label:04x}\n")

# ==============================
# 7. Done message
# ==============================
print("✅ Files generated:")
print("- weights/weights.mem")
print("- weights/biases.mem")
print("- weights/test_data.mem")

✅ Files generated:
- weights/weights.mem
- weights/biases.mem
- weights/test_data.mem
